# Remaining experiments of the 130M matched study (Kaggle, GPU T4 x2)

Continues the study from the results already published in the repository; finished runs are never repeated.

**Settings (right-hand panel):** Accelerator **GPU T4 x2**, Internet **On**. Run with *Save Version → Save & Run All (Commit)*.

**Stages, in order of importance**

| Stage | What it adds | Runs | Rough GPU time |
|---|---|---|---|
| A | third seed of the constrained adversarial arm | 1 | 2 h |
| B | **tightened constants**: bounds closed gradually during fine-tuning until every block has the per-token retention factor of the paper's constants ($\rho_{\min}=e^{-0.5}=0.61$), clean and adversarial, 3 seeds | 6 | 10 to 12 h |
| C | **paper constants, longer fine-tuning** (2000 steps instead of 400), clean and adversarial, seed 0 | 2 | 5 to 6 h |
| D | single-constraint ablations for 3 seeds | 12 | 16 to 18 h |
| E | intermediate tightening ($\Delta_{\max}\le0.5$, $\lambda_{\max}\le16$), seed 0 | 2 | 4 h |

With two GPUs the wall-clock time is about half the GPU time. One session (9-hour budget) covers roughly A, B and C. **To continue in a later session:** open this notebook, *Add Input → Your Work →* the finished version of this notebook, and run again. Stages that are complete are skipped.

Not in this notebook: 370M parameters (does not fit a 15 GB T4 in reasonable time; use an A100 with `--preset a100-370m --base state-spaces/mamba-370m-hf`), and RoBench-25 / HarmBench (they need the benchmark data and a judge model).

In [ ]:
import os, time, glob, shutil, torch
T0 = time.time()
BUDGET_HOURS = 9.0          # no new run starts after this, so a 12-hour session finishes and saves its output
STAGES = "ABCDE"            # e.g. "DE" to run only those stages
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4 x2"
n_gpu = torch.cuda.device_count(); mem = torch.cuda.get_device_properties(0).total_memory / 2**30
PRESET = "t4" if mem < 20 else ("l4" if mem < 30 else "a100")
print(f"{n_gpu} x {torch.cuda.get_device_name(0)} ({mem:.0f} GB) -> preset {PRESET}")
OUT = "/kaggle/working/lipmamba_scale_study"; os.makedirs(OUT + "/main", exist_ok=True)

## 1. Code, earlier results and checks

In [ ]:
%cd /kaggle/working
!rm -rf CV && git clone -q --depth 1 -b claude/implement-lipmamba-model-SHbBY https://github.com/rogerpanel/CV.git
%cd /kaggle/working/CV/lipmamba
!pip -q install transformers datasets pytest
os.environ["PYTHONPATH"] = "/kaggle/working/CV/lipmamba/src"
!git log --oneline -1
# finished runs: from the repository (first Kaggle session) and from any earlier version of this notebook added as input
sources = glob.glob("paper/scale_study/kaggle_t4/*_s[0-9].json") + glob.glob("/kaggle/input/**/lipmamba_scale_study/main/*_s[0-9].json", recursive=True)
for f in sources:
    if not os.path.exists(f"{OUT}/main/{os.path.basename(f)}"):
        shutil.copy(f, OUT + "/main/")
print(len(glob.glob(OUT + "/main/*_s[0-9].json")), "finished runs available")
!python -m pytest -q -o addopts="" tests/test_pretrained.py tests/test_scale_study.py tests/test_fast_scan.py 2>&1 | tail -1

## 2. The stages
Progress lines appear every five minutes; full logs are in `lipmamba_scale_study/main/logs`. A final line starting with `!!!!` means some runs failed.

In [ ]:
def left():
    return f"{max(0.01, BUDGET_HOURS - (time.time() - T0) / 3600):.2f}"
ALL = f"python scripts/scale_study.py all --dir {OUT}/main --jobs {n_gpu}"
if "A" in STAGES:
    rem = left()
    !$ALL --seeds 0 1 2 --time-budget-hours $rem -- --preset $PRESET
if "B" in STAGES:
    rem = left()
    !$ALL --seeds 0 1 2 --models lipcal lipcal_at --time-budget-hours $rem -- --preset $PRESET --tag _tight --delta-cap 0.125 --lambda-cap 4
if "C" in STAGES:
    rem = left()
    !$ALL --seeds 0 --models lip lip_at --time-budget-hours $rem -- --preset $PRESET --tag _long --steps 2000
if "D" in STAGES:
    rem = left()
    !$ALL --seeds 0 1 2 --models lipcal_no_sn lipcal_no_clamp lipcal_free_A lipcal_no_clip --time-budget-hours $rem -- --preset $PRESET
if "E" in STAGES:
    rem = left()
    !$ALL --seeds 0 --models lipcal lipcal_at --time-budget-hours $rem -- --preset $PRESET --tag _mid --delta-cap 0.5 --lambda-cap 16

## 3. Summary and results file
Download `lipmamba_scale_results.zip` from the Output tab and upload it to the chat. In the table, `rho_min_worst_block` in `summary.json` is the per-token retention factor the constraints guarantee.

In [ ]:
!python scripts/scale_study.py aggregate --dir $OUT/main
!rm -f $OUT/main/*.ckpt.pt
!cd $OUT && zip -qr /kaggle/working/lipmamba_scale_results.zip main -x '*.pt'
!rm -rf /kaggle/working/CV
!ls -la /kaggle/working